# 2. Chunk and index

Reads the parsed-docs Delta table from **notebook 1**, chunks each document with a character-based splitter, writes a chunked Delta table (with CDC enabled), and creates a Vector Search index using Delta Sync.

**Prerequisite:** Run `01-ingest-parse-pdfs.ipynb` and set `PARSED_TABLE` below to match that notebook's output table.

**Output:** A chunked Delta table (`chunk_id`, `content_chunked`, `doc_uri`, …) and a Vector Search index on the same table.

## Config

Use the same catalog/schema as notebook 1. Set the parsed table name to the output of notebook 1. Configure chunked table name, vector search endpoint, and embedding model endpoint.

In [0]:
CATALOG = "main"
SCHEMA = "default"

PARSED_TABLE = f"{CATALOG}.{SCHEMA}.rag_parsed_docs"
CHUNKED_TABLE = f"{CATALOG}.{SCHEMA}.rag_docs_chunked"
VECTOR_INDEX_NAME = f"{CATALOG}.{SCHEMA}.rag_docs_chunked_index"

VECTOR_SEARCH_ENDPOINT = "one-env-shared-endpoint"
EMBEDDING_ENDPOINT = "databricks-bge-m3"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

print(f"Parsed table: {PARSED_TABLE}")
print(f"Chunked table: {CHUNKED_TABLE}")
print(f"Vector index: {VECTOR_INDEX_NAME}")

## Install chunking dependency

We use character-based chunking (no tokenizer) so no embedding-model dependency is required for this step.

In [0]:
%pip install langchain-text-splitters --quiet
dbutils.library.restartPython()

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunk_text(doc: str) -> list:
    if not doc or not doc.strip():
        return []
    return splitter.split_text(doc)

chunk_udf = F.udf(chunk_text, ArrayType(StringType()), useArrow=True)

parsed_df = spark.table(PARSED_TABLE)
propagate_columns = ["doc_uri", "parser_status", "last_modified"]
doc_column = "content"

chunked_df = (
    parsed_df.withColumn("content_chunked", chunk_udf(F.col(doc_column)))
    .select(*propagate_columns, F.explode("content_chunked").alias("content_chunked"))
    .withColumn("chunk_id", F.md5(F.col("content_chunked")))
    .select("chunk_id", "content_chunked", *propagate_columns)
)

chunked_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CHUNKED_TABLE)
n_chunks = spark.table(CHUNKED_TABLE).count()
print(f"Wrote {n_chunks} chunks to {CHUNKED_TABLE}")

spark.sql(f"ALTER TABLE {CHUNKED_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("CDC enabled for Vector Search sync.")
display(spark.table(CHUNKED_TABLE))

## Create Vector Search index (Delta Sync)

Creates a vector index on the chunked table. Embeddings are computed by the specified embedding endpoint. Index name and endpoint must already exist or be created in the workspace.

In [0]:
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
    VectorIndexType,
)
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors.platform import ResourceDoesNotExist, BadRequest
import time

def build_retriever_index(
    vector_search_endpoint: str,
    chunked_docs_table_name: str,
    vector_search_index_name: str,
    embedding_endpoint_name: str,
    force_delete_index_before_create: bool = False,
    primary_key: str = "chunk_id",
    embedding_source_column: str = "content_chunked",
) -> tuple:
    w = WorkspaceClient()
    vsc = w.vector_search_indexes

    def find_index(name):
        try:
            return vsc.get_index(index_name=name)
        except ResourceDoesNotExist:
            return None

    existing = find_index(vector_search_index_name)
    if existing:
        if force_delete_index_before_create:
            vsc.delete_index(index_name=vector_search_index_name)
            while find_index(vector_search_index_name):
                time.sleep(30)
            create_index = True
        else:
            while not existing.status.ready:
                print("Index not ready, waiting 30s...")
                time.sleep(30)
                existing = find_index(vector_search_index_name)
            try:
                vsc.sync_index(index_name=vector_search_index_name)
                return (False, f"Sync started for {vector_search_index_name}")
            except BadRequest as e:
                return (True, f"Sync already in progress or error: {e}")
    else:
        create_index = True

    if create_index:
        delta_sync_spec = DeltaSyncVectorIndexSpecRequest(
            source_table=chunked_docs_table_name,
            pipeline_type=PipelineType.TRIGGERED,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name=embedding_source_column,
                    embedding_model_endpoint_name=embedding_endpoint_name,
                )
            ],
        )
        vsc.create_index(
            name=vector_search_index_name,
            endpoint_name=vector_search_endpoint,
            primary_key=primary_key,
            index_type=VectorIndexType.DELTA_SYNC,
            delta_sync_index_spec=delta_sync_spec,
        )
        return (False, f"Created index {vector_search_index_name}")
    return (False, "OK")

is_error, msg = build_retriever_index(
    vector_search_endpoint=VECTOR_SEARCH_ENDPOINT,
    chunked_docs_table_name=CHUNKED_TABLE,
    vector_search_index_name=VECTOR_INDEX_NAME,
    embedding_endpoint_name=EMBEDDING_ENDPOINT,
    force_delete_index_before_create=False,
)
if is_error:
    raise Exception(msg)
print(msg)
print("Note: Index may still be syncing. Check the Vector Search UI for status.")